In [ ]:
from google.colab import drive
drive.mount( '/content/gdrive' )

Mounted at /content/gdrive


# Import Library

In [ ]:
#Các hàm cần thiết để tính toán
import numpy as np
from numpy import pi
import matplotlib.pyplot as plt
import csv 
import math
import cv2
import os

In [ ]:
def calR(x,y,z,wc,lc,dc,delta,xx,zz):
    # calR = Exp(zz / sicma) * (z - zz) * (1 / ((x - xx) ^ 2 + (y + l / 2) ^ 2 + (z - zz) ^ 2) ^ 1.5 - 1 / ((x - xx) ^ 2 + (y - l / 2) ^ 2 + (z - zz) ^ 2) ^ 1.5)  ' case 1
    out = 1/4/pi * math.exp(zz/delta)*(z-zz) * (1 / ( ((x-xx/2)**2 + (y+lc/2)**2 + (z-zz)**2)**1.5 ) - 1 /  (((x-xx/2)**2 + (y-lc/2)**2 + (z-zz)**2)**1.5 ))
    return out

def calR_add1(x,y,z,wc,lc,dc,delta,yy,zz):
    # calR_add1 = Exp(zz / sicma) * (z - zz) * ((-yy - l / 2 + d) / d) * (1 / ((x + w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5 + 1 / ((x - w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5) ' case 3
    out = 1/4/pi * math.exp(zz/delta)*(z-zz)*(-yy-lc/2+dc)/dc * (1 / ( ((x+wc/2)**2 + (y-yy)**2 + (z-zz)**2)**1.5 ) + 1 /  (((x-wc/2)**2 + (y-yy)**2 + (z-zz)**2)**1.5 ))
    return out

def calR_add2(x,y,z,wc,lc,dc,delta,yy,zz):
    # calR_add2 = Exp(zz / sicma) * (z - zz) * ((yy - l / 2 + d) / d) * (1 / ((x + w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5 + 1 / ((x - w / 2) ^ 2 + (y - yy) ^ 2 + (z - zz) ^ 2) ^ 1.5) ' case 3	
    out = 1/4/pi * math.exp(zz/delta)*(z-zz)*(yy-lc/2+dc)/dc * (1 / ( ((x+wc/2)**2 + (y-yy)**2 + (z-zz)**2)**1.5 ) + 1 /  (((x-wc/2)**2 + (y-yy)**2 + (z-zz)**2)**1.5 ))
    return out

def InteR(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = -wc / 2
    b = wc / 2
    c = -dc
    d = 0
    
    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0
    
    T1 = calR(x, y, z, wc, lc, dc, delta, a, c) + calR(x, y, z, wc, lc, dc, delta, b, c) + calR(x, y, z, wc, lc, dc, delta, a, d) + calR(x, y, z, wc, lc, dc, delta, b, d)
    
    for j in range(1,n):
        yj = c + j * h
        T3 = T3 + calR(x, y, z, wc, lc, dc, delta, a, yj) + calR(x, y, z, wc, lc, dc, delta, b, yj)

    for i in range(1,m):
        xi = a + i * k
        T2 = T2 + calR(x, y, z, wc, lc, dc, delta, xi, c) + calR(x, y, z, wc, lc, dc, delta, xi, d)
        for j in range(1,n):
            yj = c + j * h
            T4 = T4 + calR(x, y, z, wc, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)
    return out

def InteR_add1(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = -lc / 2
    b = -lc / 2 + dc
    c = -dc
    d = 0
    
    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0
    
    T1 = calR_add1(x, y, z, wc, lc, dc, delta, a, c) + calR_add1(x, y, z, wc, lc, dc, delta, b, c) + calR_add1(x, y, z, wc, lc, dc, delta, a, d) + calR_add1(x, y, z, wc, lc, dc, delta, b, d)
    
    for j in range(1,n):
        yj = c + j * h
        T3 = T3 + calR_add1(x, y, z, wc, lc, dc, delta, a, yj) + calR_add1(x, y, z, wc, lc, dc, delta, b, yj)

    for i in range(1,m):
        xi = a + i * k
        T2 = T2 + calR_add1(x, y, z, wc, lc, dc, delta, xi, c) + calR_add1(x, y, z, wc, lc, dc, delta, xi, d)
        for j in range(1,n):
            yj = c + j * h
            T4 = T4 + calR_add1(x, y, z, wc, lc, dc, delta, xi, yj)
    
    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)

    return out

def InteR_add2(x, y, z, wc, lc, dc, delta):
    n = 20
    m = 20

    a = lc / 2 - dc
    b = lc / 2
    c = -dc
    d = 0
    
    h = (d - c) / n
    k = (b - a) / m

    T2 = 0
    T3 = 0
    T4 = 0
    
    T1 = calR_add2(x, y, z, wc, lc, dc, delta, a, c) + calR_add2(x, y, z, wc, lc, dc, delta, b, c) + calR_add2(x, y, z, wc, lc, dc, delta, a, d) + calR_add2(x, y, z, wc, lc, dc, delta, b, d)
    
    for j in range(1,n):
        yj = c + j * h
        T3 = T3 + calR_add2(x, y, z, wc, lc, dc, delta, a, yj) + calR_add2(x, y, z, wc, lc, dc, delta, b, yj)

    for i in range(1,m):
        xi = a + i * k
        T2 = T2 + calR_add2(x, y, z, wc, lc, dc, delta, xi, c) + calR_add2(x, y, z, wc, lc, dc, delta, xi, d)
        for j in range(1,n):
            yj = c + j * h
            T4 = T4 + calR_add2(x, y, z, wc, lc, dc, delta, xi, yj)

    out = h * k * (T1 / 4 + T2 / 2 + T3 / 2 + T4)
    
    return out

def transform(x,y,x_offset,y_offset,angle):
    xr = (x - x_offset) * math.cos(angle*math.pi/180) - (y - y_offset) * math.sin(angle*math.pi/180)
    yr = (x - x_offset) * math.sin(angle*math.pi/180) + (y - y_offset) * math.cos(angle*math.pi/180)
    return xr, yr

In [ ]:
wc_all = np.linspace(0.3,1.5,12)
lc_all = np.linspace(2,20,19)
dc_all = np.linspace(0.1,3,29)

In [ ]:
import os
import csv
# path = "/content/gdrive/MyDrive/Dipole Model/Data/Low Data"
path1 = "/content/gdrive/MyDrive/Dipole Model/Data/Low Data"

# **Tạo ra tập dataset**

In [ ]:
N = 16
Res = 0.78  # Resolution mm
aR = np.linspace(-N * Res, N * Res, 2*N)
bR = np.linspace(-N * Res, N * Res, 2*N)
H = np.zeros((len(aR), len(bR)))
x_offset=0
y_offset=0
angle=90
# wc=0.7
# lc=10
# dc=3
z=1
freq=5000
sicma=35461000
mu=0.0000012566
csi=0.085
K=1.5
I=0.01
G=3981

delta = 1 / np.sqrt(pi * freq * mu * sicma) * 1000

for wc in wc_all:
  for lc in lc_all:
    for dc in dc_all:
      tam = 'Rectangular' + '_' + str(wc) + '_' + str(lc) + '_' + str(dc) + '.csv'
      if(tam not in  os.listdir(os.path.expanduser(path1))):
        print(tam)
        H = np.zeros((len(aR), len(bR)))
        for u, xo in enumerate(aR):
          for v, yo in enumerate(aR):
            x, y = transform(xo, yo, x_offset, y_offset, angle)
            H[u, v] = InteR(x, y, z, wc, lc, dc, delta) + InteR_add1(x, y, z, wc, lc, dc, delta) - InteR_add2(x, y, z, wc, lc, dc, delta)
            H[u, v] = math.cos(angle * math.pi / 180) * H[u, v]

        H = csi*H
        H = K*I*G*H

        name_params=['N','liff_off','Csi','K','I','f','sicma','mu','G','Res']
        params=[32, z, csi, K, I, freq, sicma, mu, G, Res]
        L = ['shape', 'width', 'length', 'depth','angle','x_offset','y_offset']
        C = ['Rectangular', wc, lc, dc, angle, x_offset, y_offset]

        filename = '/content/gdrive/MyDrive/Dipole Model/Data/Low Data/' + C[0] + '_' + str(C[1]) + '_' + str(C[2]) + '_' + str(C[3]) + '.csv'

        with open(filename, 'w', newline='') as f:
          writer = csv.writer(f)
          writer.writerow(name_params)
          writer.writerow(params)
          writer.writerow(L)
          writer.writerow(C)
          writer.writerows(H)
          f.close()

#0.40909090909090906 16.0 0.1
#0.3 16.0 2.1714285714285713

Rectangular_1.5_8.0_1.9642857142857142.csv
Rectangular_1.5_8.0_2.0678571428571426.csv
Rectangular_1.5_8.0_2.1714285714285713.csv
Rectangular_1.5_8.0_2.275.csv
Rectangular_1.5_8.0_2.3785714285714286.csv
Rectangular_1.5_8.0_2.482142857142857.csv
Rectangular_1.5_8.0_2.585714285714286.csv
Rectangular_1.5_8.0_2.689285714285714.csv
Rectangular_1.5_8.0_2.7928571428571427.csv
Rectangular_1.5_8.0_2.8964285714285714.csv
Rectangular_1.5_8.0_3.0.csv
Rectangular_1.5_9.0_0.1.csv
Rectangular_1.5_9.0_0.20357142857142857.csv
Rectangular_1.5_9.0_0.30714285714285716.csv
Rectangular_1.5_9.0_0.4107142857142857.csv
Rectangular_1.5_9.0_0.5142857142857142.csv
Rectangular_1.5_9.0_0.6178571428571428.csv
Rectangular_1.5_9.0_0.7214285714285714.csv
Rectangular_1.5_9.0_0.825.csv
Rectangular_1.5_9.0_0.9285714285714285.csv
Rectangular_1.5_9.0_1.032142857142857.csv
Rectangular_1.5_9.0_1.1357142857142857.csv
Rectangular_1.5_9.0_1.2392857142857143.csv
Rectangular_1.5_9.0_1.342857142857143.csv
Rectangular_1.5_9.0_1.44642